# <center>**Travaux exploratoires : résumé formaté d'un texte</center>**

# <center>**IV. Extraction structurée avec le modèle llama3-70b-8192</center>**

*Ce fichier est généré sur Jupyter. Pour le faire fonctinner, il faut se placer dans l'environemment coreferee-env.*

*Date de dernière mise à jour : 30/07/2025*

**Méthodologie :**

- prompter directement un modèle réputé fiable : le modèle llama3-70b-8192
- le prompt se fait via une API
- on intègre un accès à l'API dans une fonction qui à un nom de fichier texte (.txt ou .csv) va associer en sortie un fichier csv et un fichier json stockant les éléments souhaités : résumé de l'événement, lieu, moment, individus.

**Résultats :**

- très intéressants d'un point de vue qualitatif, l'extraction structurée de textes est vraiment très correcte
- le temps d'exécution est correct : environ 2 secondes pour un fichier de 50 lignes
- le principal défaut de cette approche est son manque de souveraineté : les données entrées s'échappent vers un serveur de la compagnie améariciane Groq, propriétaire de l'API utiliséee ici pour accéder au modèle llama3-70b-8192
- une autre limitation est le nombre de tokens : le modèle a été testé sur un recueil de nouvelles de 33 pages. Ce volume s'est avéré trop important. Cette approche ne fonctionne donc plus de façon directe et il faut procéder à un prétraitement du texte qui consiste à le découper en blocs, puis à appliquer le modèle à chaque bloc
- ce modèle est plus efficace que Flan-T5 dans la tache d'extraction structurée
- il est aussi plus lourd, du fait d'un nombre beaucoup plus élevé de paramètres

**Améliorations possibles :**

- il est *a priori* possible d'utiliser le modèle en local, donc sans API, ce qui règlerait le problème principal à cette approche. Cette solution est à essayer, mais il y a peu de chances qu'elle soit viable sans GPU.

- voir si d'autes solutions gratuites existent :

  - soit avec hébergement uniquement sur des serveurs français
  - soit avec d'autres modèles qui pourraient tourner uniquement en local, qui ne nécessiteraient pas de GPU pour tourner en un temps raisonnable, et qui fourniraient des résultats qualitatifs (cela semnble compliqué !)

**Conclusion :**

- malgré le défaut principal sur la souveraineté, ce modèle est intéressant à la fois par la qualité des résulutats produits et par son temps d'exécution
- une fonction visant à découper un textes en blocs est implémentée dans un prochain notebook. Elle sera combinée à la fonction codée ici pour étendre l'extraction à une volumétrie plus importante des textes en entrée.


## **1. Aperçu technique**

**Modèle gratuit mais données probablement hébergées aux Etats-Unis :**

- modèle open source
- développé par **Meta**
- hébergé et rendu accessible gratuitement par l'**API Groq**
- cette API héberge des modèles open source comme LLaMa3 et Mixtral
- elle est complètement gratuite (en juillet 2025), rapide et stable
- les données entrées dans le modèle vont sur le serveur de Groq, qui est une entreprise américaine, donc le serveur est probablement situé aux Etats-Unis
- un appel à la bibliothèque client openai, mais uniquement pour interagir avec une API : le modèle n'est pas un modèle OpenAI, les données ne sont pas hébergées sur un serveur OpenAI
- **taille du contexte** du modèle llama3-70b-8192 : 8 192 tokens

| Élément                      | Rôle réel                                                                                                  |
| ---------------------------- | ---------------------------------------------------------------------------------------------------------- |
| 🧠 **OpenAI**                | Crée **GPT-3.5**, **GPT-4**, **ChatGPT**, etc. — modèles propriétaires                                     |
| 🐍 **`openai` (lib Python)** | Une **bibliothèque client** pour interagir avec une API **OpenAI-compatible** (même structure de requêtes) |
| ⚙️ **Groq**                  | Fournisseur d’infrastructure qui héberge **des modèles open-source** comme **LLaMA 3** et **Mixtral**      |
| 🦙 **LLaMA 3**               | Modèle **développé par Meta**, totalement **open-source**  

**Comment ça marche, ici ?**

✅ Tu utilises la bibliothèque openai pour envoyer des requêtes...

✅ ...mais tu n’interroges pas OpenAI, tu interroges l’API de Groq

✅ Groq héberge et te donne accès à des modèles open-source

✅ Le modèle que tu utilises (llama3-70b-8192) vient de Meta, pas d’OpenAI

**Perspectives :**

- tester une version complètement locale (hors-ligne) avec llama.cpp ou Ollama
- continuer avec Groq mais en ajoutant des mesures de confidentialité (anonymisation, filtrage, etc.) dans tes textes ?


## **2. Un exemple d'utilisation sur un texte très court**

**Bilan : plutôt convaincant**

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="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",
    base_url="https://api.groq.com/openai/v1",
    default_headers={"User-Agent": "PythonClient/1.0"}  # sécurisé
)

texte = "Le corps a été retrouvé dans un puits sec au lever du jour. Aucun témoin n'était présent."

prompt = f"""
Voici un texte :
{texte}

Tu dois extraire chaque fait important du texte sous forme d’une structure JSON avec ces champs :
- résumé : résumé en 5-6 mots
- lieu : lieu s'il y en a un, sinon "NA"
- moment : moment s'il y en a un, sinon "NA"
- individus : liste des individus impliqués s’il y en a, sinon "NA"

Retourne uniquement la liste JSON, rien d’autre.
"""

response = client.chat.completions.create(
    model="llama3-70b-8192",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)


[
  {
    "résumé": "Corps retrouvé dans puits",
    "lieu": "puits sec",
    "moment": "lever du jour",
    "individus": "NA"
  }
]


## **3. Construction d'un pipeline complet**

**But :** à partir d’un fichier CSV avec une colonne texte :

    - lire le fichier CSV

    - envoyer chaque texte au modèle llama3-70b-8192 via Groq

    - extraire pour chaque fait :

        - résumé (5-6 mots)

        - lieu

        - moment

        - individus

    - écrire les résultats dans un nouveau fichier CSV

In [ ]:
import pandas as pd
from openai import OpenAI
import time

# 🔑 Remplace par ta clé Groq
GROQ_API_KEY = "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"

# 📍 Configuration du client OpenAI pour l'API Groq
client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    default_headers={"User-Agent": "pipeline-faits/1.0"}
)

# 🧠 Fonction d'extraction à partir d’un texte
def extraire_faits(texte):
    prompt = f"""
Voici un texte :
{texte}

Tu dois extraire chaque fait important du texte sous forme d’une structure JSON avec ces champs :
- résumé : résumé du fait en 5 ou 6 mots
- lieu : lieu s'il y en a un, sinon "NA"
- moment : moment s'il y en a un, sinon "NA"
- individus : liste des individus impliqués s’il y en a, sinon "NA"

Retourne uniquement la liste JSON, sans explication.
"""
    try:
        response = client.chat.completions.create(
            model="llama3-70b-8192",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Erreur : {e}")
        return "ERREUR"

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving fichier_textes.csv to fichier_textes.csv


In [ ]:
# 📂 Chargement du fichier CSV
df = pd.read_csv("fichier_textes.csv")  # ← change le nom ici

# 🧮 Application du modèle
df["faits_structurés"] = df["texte"].apply(lambda x: extraire_faits(x) if isinstance(x, str) and x.strip() else "NA")
time.sleep(1)  # (optionnel) pour ralentir si quota

# 💾 Sauvegarde du résultat
df.to_csv("resultats_structures.csv", index=False)
print("✅ Résultats enregistrés dans 'resultats_structures.csv'")

✅ Résultats enregistrés dans 'resultats_structures.csv'


In [ ]:
df.head()

,texte,faits_structurés
0,le gardien du parc a vu une vitre brisée près ...,"[\n {\n ""résumé"": ""Vitre brisée près du po..."
1,un inconnu cagoulé a découvert un sac abandonn...,"[\n {\n ""résumé"": ""Découverte d'un sac aba..."
2,le gardien du parc a entendu des cris inhabitu...,"[\n {\n ""résumé"": ""Cris inhabituels entend..."
3,un inconnu cagoulé a découvert un sac abandonn...,"[\n {\n ""résumé"": ""Sac abandonné trouvé"",\..."
4,le gardien du parc a remarqué un comportement ...,"[\n {\n ""résumé"": ""Comportement suspect ob..."


**On le convertit en fichier json**

In [ ]:
import pandas as pd
import json

# 📥 Chargement du fichier CSV avec les faits structurés en texte
df = pd.read_csv("resultats_structures.csv")  # ← à adapter selon ton fichier

# 📦 Création de la liste d'objets JSON
json_liste = []

for _, row in df.iterrows():
    texte = row["texte"]
    try:
        # On suppose que "faits_structurés" est une chaîne JSON contenant une liste de faits
        faits_structures = json.loads(row["faits_structurés"])
        for fait in faits_structures:
            json_liste.append({
                "texte": texte,
                "résumé": fait.get("résumé", "NA"),
                "lieu": fait.get("lieu", "NA"),
                "moment": fait.get("moment", "NA"),
                "individus": fait.get("individus", "NA")
            })
    except Exception as e:
        # En cas d'erreur de parsing, on insère une ligne neutre
        json_liste.append({
            "texte": texte,
            "résumé": "ERREUR",
            "lieu": "NA",
            "moment": "NA",
            "individus": "NA"
        })

# 💾 Sauvegarde dans un fichier JSON
with open("fichier_faits_structures.json", "w", encoding="utf-8") as f:
    json.dump(json_liste, f, indent=2, ensure_ascii=False)

print("✅ Fichier JSON structuré enregistré sous 'fichier_faits_structures.json'")

✅ Fichier JSON structuré enregistré sous 'fichier_faits_structures.json'


In [ ]:
import json

with open("fichier_faits_structures.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Exemple : afficher les 3 premiers éléments
for ligne in data[:3]:
    print(ligne)

{'texte': 'le gardien du parc a vu une vitre brisée près du pont de Neuilly à l’aube.', 'résumé': 'Vitre brisée près du pont', 'lieu': 'Pont de Neuilly', 'moment': "à l'aube", 'individus': ['le gardien du parc']}
{'texte': 'un inconnu cagoulé a découvert un sac abandonné dans un entrepôt désaffecté peu avant l’ouverture des bureaux.', 'résumé': "Découverte d'un sac abandonné", 'lieu': 'entrepôt désaffecté', 'moment': "peu avant l'ouverture des bureaux", 'individus': ['un inconnu cagoulé']}
{'texte': 'le gardien du parc a entendu des cris inhabituels sur le quai de la ligne 12 vers 22h.', 'résumé': 'Cris inhabituels entendus', 'lieu': 'quai de la ligne 12', 'moment': 'vers 22h', 'individus': ['le gardien du parc']}


**Bilan :**

- plutôt satisfaisant sur les exemples traités (exemples générés par ChatGPT)
- le résumé semble être une simple extraction du passage le plus significatif du texte

**Pour la suite :**

- construction d'une fonction qui construit l'ensemble du pipeline
- essais avec des textes rédigés à la main

# **4. Fonction d'extraction structurée**

**Un rappel :** la fonction clé est la fonction *extraire_faits()* que l'on rappelle ici

**4.a. Configuration du client OpenAI pour l'API Groq**

In [ ]:
import pandas as pd
from openai import OpenAI
import time

# 🔑 Remplace par ta clé Groq
GROQ_API_KEY = "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"

# 📍 Configuration du client OpenAI pour l'API Groq
client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    default_headers={"User-Agent": "pipeline-faits/1.0"}
)

**4.b. Implémentation de la fonction *extraire_faits()* qui fait appel au modèle llama3-70b-8192**

**Description de cette fonction :**

- **input :**

  - le nom du fichier CSV à importer (une chaine de caractères). Ce fichier CSV est constitué d'une seule colonne qui stocke tous les textes pour lesquels on demande une extraction structurée

  - cette colonne doit être nommée "texte" (première ligne du fichier)



- **output :**

  - construction et sauvegarde d'un fichier CSV enrichi d'une colonne "faits_structurés"

  - construction d'un fichier JSON qui reprend tous les éléments de ce fichier CSV

  - affichage des 5 premières lignes de ce fichier JSON


**Amélioration possible :**

- augmenter la flexibilité de la fonction en mettant le choix du modèle en input

In [ ]:
def extraire_faits(texte):
    prompt = f"""
Voici un texte :
{texte}

Tu dois extraire chaque fait important du texte sous forme d’une structure JSON avec ces champs :
- résumé : résumé du fait en 5 ou 6 mots
- lieu : lieu s'il y en a un, sinon "NA"
- moment : moment s'il y en a un, sinon "NA"
- individus : liste des individus impliqués s’il y en a, sinon "NA"

Retourne uniquement la liste JSON, sans explication.
"""
    try:
        response = client.chat.completions.create(
            model="llama3-70b-8192",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Erreur : {e}")
        return "ERREUR"

**4.c. Implémentation de la fonction extraction_structuree()**

**Description de cette fonction :**

- **input :** cette fonction prend en charge deux formats de fichiers :
  - le format CSV pour un tableau d'une seule colonne nommée *texte*
  - le format .txt

- **output :**

  - extraction pour chaque texte des faits importants, puis pour chaque fait :

     - résumé (5-6 mots)

     - lieu

     - moment

     - individus

  - écriture des résultats dans un nouveau fichier CSV et un fichier JSON

**Remarques :**

  - cette fonction n'est rien d'autre qu'une surcouche de la fonction *extraire_faits()*
  - on peut la rendre plus flexible en laissant à l'utilisateur le choix du modèle


In [ ]:
import pandas as pd
import json
import os

def extraction_structuree(nom_fichier):
    import pandas as pd
    import json
    import time
    import os

    extension = os.path.splitext(nom_fichier)[-1].lower()
    print(f"Extension détectée : {extension}")

    if extension == ".csv":
        df = pd.read_csv(nom_fichier)
        if "texte" not in df.columns:
            raise ValueError("Le fichier CSV doit contenir une colonne 'texte'")

    elif extension == ".txt":
        with open(nom_fichier, "r", encoding="utf-8") as f:
            texte = f.read()
        df = pd.DataFrame({"texte": [texte]})

    else:
        raise ValueError("Format de fichier non pris en charge. Utilisez .csv ou .txt")

    # 🧠 Application du modèle (llama3-70b-8192)
    df["faits_structurés"] = df["texte"].apply(
        lambda x: extraire_faits(x) if isinstance(x, str) and x.strip() else "NA"
    )
    time.sleep(1)

    # 💾 Sauvegarde CSV
    df.to_csv("resultats_structures.csv", index=False)
    print("✅ Résultats enregistrés dans 'resultats_structures.csv'")

    # 📦 Génération JSON structuré
    json_liste = []
    for _, row in df.iterrows():
        texte = row["texte"]
        try:
            faits_structures = json.loads(row["faits_structurés"])
            for fait in faits_structures:
                json_liste.append({
                    "texte": texte,
                    "résumé": fait.get("résumé", "NA"),
                    "lieu": fait.get("lieu", "NA"),
                    "moment": fait.get("moment", "NA"),
                    "individus": fait.get("individus", "NA")
                })
        except Exception as e:
            json_liste.append({
                "texte": texte,
                "résumé": "ERREUR",
                "lieu": "NA",
                "moment": "NA",
                "individus": "NA"
            })

    # 💾 Sauvegarde JSON
    with open("fichier_faits_structures.json", "w", encoding="utf-8") as f:
        json.dump(json_liste, f, indent=2, ensure_ascii=False)

    print("✅ Fichier JSON structuré enregistré sous 'fichier_faits_structures.json'")

    # 🔎 Aperçu
    with open("fichier_faits_structures.json", "r", encoding="utf-8") as f:
        data = json.load(f)
    for ligne in data[:3]:
        print(ligne)


# **5. Tests de validation de la fonction *extraction_structuree()***

**5.a. Test sur un fichier de petits textes *fichier_textes.csv* générés par ChatGPT**

In [ ]:
import time

start_time = time.time()

extraction_structuree("fichier_textes.csv")

end_time = time.time()
print(f"⏱️ Temps d'exécution : {end_time - start_time:.2f} secondes")

NameError: name 'extraction_structuree' is not defined

**Bilan :**

- bien d'un point de vue quantitatif
- temps d'exécution long sans GPU : presque 1 min, pour un fichier CSV de 50 lignes, où chaque ligne est constituée d'une phrase
- temps d'exécution avec GPU T4 :

**5.b. Test sur un petit fichier de petits textes écrits à la main**

In [ ]:
from google.colab import files

uploaded = files.upload() # on upolad le fichier fichier_textes_naturels.csv

In [ ]:
import time

start_time = time.time()

extraction_structuree("fichier_textes_naturels.csv")

end_time = time.time()
print(f"⏱️ Temps d'exécution : {end_time - start_time:.2f} secondes")

Extension détectée : .csv
✅ Résultats enregistrés dans 'resultats_structures.csv'
✅ Fichier JSON structuré enregistré sous 'fichier_faits_structures.json'
{'texte': 'A 14h30 ce vendredi 2 janvier, un incendie s’est déclaré près de l’entrepôt de bois du petit village de Vendée.', 'résumé': "Incendie près d'entrepôt de bois", 'lieu': 'Vendée', 'moment': '14h30, vendredi 2 janvier', 'individus': 'NA'}
{'texte': 'Léa et Wilfried se sont retrouvés à l’aéroport d’Orly hier à 20h45,', 'résumé': 'Léa et Wilfried se retrouvent', 'lieu': "aéroport d'Orly", 'moment': '20h45', 'individus': ['Léa', 'Wilfried']}
{'texte': 'L’agence BNP a été braquée ce matin aux alentours de 10h15. Deux individus armés et cagoulés ont fait irruption dans l’agence. Ils en sont ressortis à 10h21,', 'résumé': 'Agence BNP braquée', 'lieu': 'Agence BNP', 'moment': '10h15', 'individus': ['2 individus armés et cagoulés']}
⏱️ Temps d'exécution : 3.16 secondes


**Bilan pour des textes courts écrits à la main :**
- pas mal du tout d'un point de vue qualitatif
- le temps de traitement sa,s GPU (environ 3 s) est long pour juste 5 textes.
  - le modèle utilisé LLaMA 3-70B est très gros, donc plus lent que du 7B
- temps d'exécution avec GPU T4 :

**5.c. Test sur un texte de 50 lignes généré aléatoirement par ChatGPT**

In [ ]:
from google.colab import files

uploaded = files.upload() # on upolad le fichier fichier_textes_naturels.csv

Saving texte_50L.txt to texte_50L.txt


In [ ]:
import time

start_time = time.time()

extraction_structuree("texte_50L.txt")

end_time = time.time()
print(f"⏱️ Temps d'exécution : {end_time - start_time:.2f} secondes")

Extension détectée : .txt
✅ Résultats enregistrés dans 'resultats_structures.csv'
✅ Fichier JSON structuré enregistré sous 'fichier_faits_structures.json'
{'texte': '\nIl a de beaux légumes, Monsieur Jean. Brillants, juteux, colorés. On a envie\nde mordre dedans à pleines dents. Il faut dire qu’il y met du cœur à les\narroser, les désherber, les bichonner comme s’il s’agissait de ses propres\nenfants.\nHélas, il n’en a pas d’enfant, Monsieur Jean. Son épouse, la gentille Colette,\nl’a quitté il y a bien longtemps. « Un jour, elle est partie comme ça, j’ai rien\ncompris », répète-t-il lorsqu’on le questionne à ce sujet. Oui, il l’a pleurée\nau début. Et puis il est passé à autre chose.\nIl s’est bien fait aguicher par la Christiane, la veuve du village, mais il n’a\npas semblé intéressé. Christiane, quant à elle, trouvait que c’était un beau\nparti, Monsieur Jean. Sans être amoureuse, elle se disait qu’être mariée\nau jardinier du château de Champs-sur-Marne, ça faisait classe ! Elle se

**Bilan :**
- satisfaisant d'un point de vue qualitatif, les éléments importants du texte ont effectivement été détectés, et bien structurés
- le temps d'exécution est cette fois beaucoup plus rapide : 2,79 s pour un texte de 50 lignes **sans GPU**
- il est probable que le nombre de textes, qui correspond exactement au nombre d'appels de la fonction extraire_faits() - soit plus coûteux en temps que la longueur d'un texte

**5.d. Test sur un compte-rendu d'enquête d'une centaine de lignes, généré aléatoirement par ChatGPT**

In [ ]:
from google.colab import files

uploaded = files.upload() # on upolad le fichier fichier_textes_naturels.csv

Saving compte_rendu_enquete.txt to compte_rendu_enquete.txt


In [ ]:
import time

start_time = time.time()

extraction_structuree("compte_rendu_enquete.txt")

end_time = time.time()
print(f"⏱️ Temps d'exécution : {end_time - start_time:.2f} secondes")

Extension détectée : .txt
✅ Résultats enregistrés dans 'resultats_structures.csv'
✅ Fichier JSON structuré enregistré sous 'fichier_faits_structures.json'
{'texte': '\nCompte-rendu d\'enquête - Affaire n°4782C - Vol aggravé avec violences\n\nService en charge : Brigade de sûreté urbaine\nDate de l\'enquête : 12 mars 2025\nLieu de l’infraction : Rue des Tilleuls, quartier de la Madeleine, Angers (49)\n\n1. Contexte général\nLe 11 mars 2025 à 22h47, la permanence du commissariat d\'Angers reçoit un appel d\'un témoin signalant une agression en cours devant le bar "Le Perroquet Bleu", rue des Tilleuls. Une patrouille est immédiatement dépêchée sur place. À leur arrivée à 22h53, les agents trouvent un homme au sol, visiblement blessé, et une femme en état de choc.\n\n2. Description de la scène\nLa victime, identifiée comme Monsieur Kévin Raguin (né le 14 août 1993 à Saint-Malo), est allongée sur le trottoir. Il présente une plaie contuse à l’arcade sourcilière droite et des éraflures multi

**Bilan :**
- satisfaisant d'un point de vue qualitatif, les éléments importants du texte ont effectivement été détectés, et bien structurés
- le temps d'exécution est cette fois beaucoup plus rapide : 2,83 s pour un texte de 50 lignes **sans GPU**
- il est probable que le nombre de textes, qui correspond exactement au nombre d'appels de la fonction extraire_faits() - soit plus coûteux en temps que la longueur d'un texte

**5.e. Test sur un recueil de nouvelles (33 pages au total)**

In [ ]:
from google.colab import files

uploaded = files.upload() # on upolad le fichier recueil_nouvelles.csv

Saving recueil_nouvelles.txt to recueil_nouvelles.txt


In [ ]:
import time

start_time = time.time()

extraction_structuree("recueil_nouvelles.txt")

end_time = time.time()
print(f"⏱️ Temps d'exécution : {end_time - start_time:.2f} secondes")

Extension détectée : .txt
Erreur : Error code: 413 - {'error': {'message': 'Request too large for model `llama3-70b-8192` in organization `org_01k1dahs9xe4hazk1k24mv1129` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 28707, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
✅ Résultats enregistrés dans 'resultats_structures.csv'
✅ Fichier JSON structuré enregistré sous 'fichier_faits_structures.json'
{'texte': 'Nouvelles à chute !\nFatale erreur ........................................................................................................................................................................................ 2\nIceberg ............................................................................................................................................................................................

**Bilan :**

- on obtient un message d'erreur qui nous signale un dépassement de la limite de tokens par minute pour le modèle llama-70b-8192 cia Groq
- le fichier .txt est donc trop long pour une seule requête

**Solution envisagée :** découper le fichier en blocs de 5 000 à 6 000 tokens, analyser chaque bloc et concaténer les résultats